# Convolutional Neural Networks – Hands‑on Exercises
This notebook contains three exercises using **TensorFlow 2**:
1. CNN for MNIST classification
2. Visualizing feature maps in a CNN
3. Building a small ResNet with residual connections

Recommended runtime: Python ≥3.9 with TensorFlow ≥2.10


## Setup

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)


---
# Exercise 1 — CNN for MNIST
Build a CNN to classify handwritten digits.

**Learning goals**:
- Understand Conv2D layers
- Understand pooling
- Compare CNN vs dense models


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

x_train = x_train[..., None]
x_test = x_test[..., None]

print(x_train.shape)


In [ ]:
plt.imshow(x_train[0], cmap='gray')
plt.title(f"Label: {y_train[0]}")
plt.axis("off")


### Build the CNN

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.1
)


In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)


---
# Exercise 2 — Visualizing CNN Feature Maps
We inspect intermediate activations to understand what the CNN learns.

In [ ]:
sample_image = x_test[0:1]

layer_outputs = [layer.output for layer in model.layers if 'conv' in layer.name]
activation_model = tf.keras.Model(inputs=model.input, outputs=layer_outputs)

activations = activation_model.predict(sample_image)


In [ ]:
for layer_activation in activations:
    n_features = layer_activation.shape[-1]
    size = layer_activation.shape[1]

    display_grid = np.zeros((size, size * n_features))

    for i in range(n_features):
        channel_image = layer_activation[0, :, :, i]
        channel_image -= channel_image.mean()
        channel_image /= (channel_image.std() + 1e-5)
        channel_image *= 64
        channel_image += 128
        channel_image = np.clip(channel_image, 0, 255).astype('uint8')
        display_grid[:, i * size : (i + 1) * size] = channel_image

    scale = 1. / size
    plt.figure(figsize=(scale * display_grid.shape[1], scale * display_grid.shape[0]))
    plt.title("Feature maps")
    plt.grid(False)
    plt.imshow(display_grid, aspect='auto', cmap='viridis')


---
# Exercise 3 — Build a Small ResNet
We implement a residual block and build a small CNN using skip connections.

**Concept**:
y = F(x) + x


In [ ]:
from tensorflow.keras import layers

def residual_block(x, filters):
    shortcut = x

    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)

    return x


In [ ]:
inputs = tf.keras.Input(shape=(28,28,1))

x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)

x = residual_block(x, 32)
x = layers.MaxPooling2D()(x)

x = residual_block(x, 32)

x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(10, activation='softmax')(x)

resnet_model = tf.keras.Model(inputs, outputs)

resnet_model.summary()


In [ ]:
resnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
resnet_model.fit(
    x_train,
    y_train,
    epochs=5,
    validation_split=0.1
)


In [ ]:
resnet_model.evaluate(x_test, y_test)


## Discussion Questions
1. Why do residual connections help training deep networks?
2. Compare the performance of the CNN and ResNet models.
3. What patterns appear in early feature maps?
4. What would happen if we removed pooling layers?
